In [1]:
!pip install onnx onnxruntime onnxscript scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 5.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import onnx
import onnxruntime as rt

In [3]:
batch_size = 64
learning_rate = 0.001
N_Epochs = 100
epsilon = 0.0001


In [4]:
housing = fetch_california_housing()
X = housing.data.astype(np.float32)
y = housing.target.astype(np.float32).reshape(-1, 1)

feature_names = housing.feature_names
print("Feature names:", feature_names)
print("X shape:", X.shape)
print("y shape:", y.shape)

Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
X shape: (20640, 8)
y shape: (20640, 1)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_tr = torch.from_numpy(X_train)
X_test_tr = torch.from_numpy(X_test)
y_train_tr = torch.from_numpy(y_train)
y_test_tr = torch.from_numpy(y_test)


In [6]:
x_means = X_train_tr.mean(0, keepdim=True)
x_deviations = X_train_tr.std(0, keepdim=True) + epsilon

y_mean = y_train_tr.mean(0, keepdim=True)
y_std = y_train_tr.std(0, keepdim=True) + epsilon

# normalize y for training stability
y_train_norm = (y_train_tr - y_mean) / y_std
y_test_norm = (y_test_tr - y_mean) / y_std

In [7]:
train_ds = TensorDataset(X_train_tr, y_train_norm)
test_ds = TensorDataset(X_test_tr, y_test_norm)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

In [8]:
class DL_Net(nn.Module):
    def __init__(self, x_means, x_deviations):
        super().__init__()
        self.x_means = x_means
        self.x_deviations = x_deviations

        self.linear1 = nn.Linear(8, 128)
        self.act1 = nn.ReLU()
        self.linear2 = nn.Linear(128, 64)
        self.act2 = nn.ReLU()
        self.linear3 = nn.Linear(64, 32)
        self.act3 = nn.ReLU()
        self.linear4 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = (x - self.x_means) / self.x_deviations
        x = self.linear1(x)
        x = self.act1(x)
        x = self.dropout(x)
        x = self.linear2(x)
        x = self.act2(x)
        x = self.linear3(x)
        x = self.act3(x)
        x = self.linear4(x)
        return x

In [9]:
def training_loop(N_Epochs, model, loss_fn, opt):
    for epoch in range(N_Epochs):
        model.train()
        running_loss = 0.0

        for xb, yb in train_dl:
            y_pred = model(xb)
            loss = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            running_loss += loss.item()

        if epoch % 10 == 0 or epoch == N_Epochs - 1:
            print(f"Epoch {epoch}: loss = {running_loss / len(train_dl):.6f}")

In [10]:
def print_metrics_function(y_true, y_pred):
    print("Mean Squared Error (MSE): %.4f" % mean_squared_error(y_true, y_pred))
    print("Root Mean Squared Error (RMSE): %.4f" % np.sqrt(mean_squared_error(y_true, y_pred)))
    print("Mean Absolute Error (MAE): %.4f" % mean_absolute_error(y_true, y_pred))
    print("R2 Score: %.4f" % r2_score(y_true, y_pred))


In [11]:
model = DL_Net(x_means, x_deviations)
opt = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = nn.MSELoss()

training_loop(N_Epochs, model, loss_fn, opt)


Epoch 0: loss = 0.455238
Epoch 10: loss = 0.236328
Epoch 20: loss = 0.213185
Epoch 30: loss = 0.198672
Epoch 40: loss = 0.191888
Epoch 50: loss = 0.184858
Epoch 60: loss = 0.178959
Epoch 70: loss = 0.174270
Epoch 80: loss = 0.169738
Epoch 90: loss = 0.165758
Epoch 99: loss = 0.162226


In [12]:
with torch.no_grad():
    model.eval()
    for x_real, y_real_norm in test_dl:
        y_pred_norm = model(x_real)

        # unnormalize predictions back to real house value scale
        y_pred = y_pred_norm * y_std + y_mean
        y_real = y_real_norm * y_std + y_mean

        y_pred_np = y_pred.cpu().numpy().flatten()
        y_real_np = y_real.cpu().numpy().flatten()

        print_metrics_function(y_real_np, y_pred_np)


Mean Squared Error (MSE): 0.2504
Root Mean Squared Error (RMSE): 0.5004
Mean Absolute Error (MAE): 0.3387
R2 Score: 0.8089


In [13]:
torch.save({
    "model_state_dict": model.state_dict(),
    "x_means": x_means,
    "x_deviations": x_deviations
}, "california_housing_model.pth")

print("Saved PyTorch model: california_housing_model.pth")


Saved PyTorch model: california_housing_model.pth


In [14]:
checkpoint = torch.load("california_housing_model.pth", map_location=torch.device("cpu"))

model2 = DL_Net(checkpoint["x_means"], checkpoint["x_deviations"])
model2.load_state_dict(checkpoint["model_state_dict"])
model2.eval()

DL_Net(
  (linear1): Linear(in_features=8, out_features=128, bias=True)
  (act1): ReLU()
  (linear2): Linear(in_features=128, out_features=64, bias=True)
  (act2): ReLU()
  (linear3): Linear(in_features=64, out_features=32, bias=True)
  (act3): ReLU()
  (linear4): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

In [18]:
dummy_input = torch.randn(1, 8)

torch.onnx.export(
    model2,
    dummy_input,
    "california_housing_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=11,
    external_data=False,
    dynamo=False
)

print("Saved single-file ONNX model: california_housing_model.onnx")

/tmp/ipykernel_28715/3970828387.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Saved single-file ONNX model: california_housing_model.onnx


In [19]:
ort_session = rt.InferenceSession("california_housing_model.onnx")
sample_input = X_test[:5].astype(np.float32)
outputs = ort_session.run(None, {"input": sample_input})

print("ONNX raw predictions (normalized scale):")
print(outputs[0].flatten())


ONNX raw predictions (normalized scale):
[-1.3206394  -0.994121    2.697733    0.3176006   0.47382998]


In [20]:
from google.colab import files
files.download("california_housing_model.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>